# 📘 Data Quality Framework
## Databricks Notebook — Build a Production Quality Engine from Scratch

**What you'll learn:**
- Six dimensions of data quality
- Build a complete expectation engine (no external libraries)
- Expectation suites, validation results, reporting
- Quality gates in pipelines (quarantine pattern)
- Anomaly detection (volume, distribution, freshness)
- Data contracts, monitoring, alerting

**No pip installs needed** — pure Python + PySpark implementation.

**Prerequisites:** Notebooks 01-26 (techmart_dw)

---

---
# 📗 Section 1: Data Quality Dimensions

| Dimension | Question | Example Check |
|---|---|---|
| **Completeness** | Are required fields populated? | email NULL rate < 10% |
| **Uniqueness** | Are keys unique? | order_id has no duplicates |
| **Validity** | Are values in expected domain? | status IN ('completed','pending',...) |
| **Accuracy** | Do values match reality? | total_amount = sum(line_items) |
| **Consistency** | No contradictions? | order_date <= ship_date |
| **Timeliness** | Is data fresh enough? | max(created_at) within 24 hours |

---
# 📗 Section 2: Building an Expectation Engine

In [ ]:
from pyspark.sql.functions import *
from datetime import datetime

class Expectation:
    """A single data quality expectation."""
    
    def __init__(self, name, check_type, column=None, params=None, severity="error"):
        self.name = name
        self.check_type = check_type
        self.column = column
        self.params = params or {}
        self.severity = severity  # error, warning, info

class ExpectationEngine:
    """Run expectations against a Spark DataFrame."""
    
    def __init__(self, table_name):
        self.table_name = table_name
        self.df = spark.table(table_name)
        self.total_rows = self.df.count()
        self.results = []
    
    def run(self, expectation):
        """Run a single expectation and record result."""
        e = expectation
        
        if e.check_type == "not_null":
            failures = self.df.filter(f"{e.column} IS NULL").count()
            pass_rate = (self.total_rows - failures) / self.total_rows
            
        elif e.check_type == "unique":
            distinct = self.df.select(e.column).distinct().count()
            failures = self.total_rows - distinct
            pass_rate = distinct / self.total_rows
            
        elif e.check_type == "in_set":
            valid_values = e.params["values"]
            failures = self.df.filter(~col(e.column).isin(valid_values)).count()
            pass_rate = (self.total_rows - failures) / self.total_rows
            
        elif e.check_type == "between":
            lo, hi = e.params["min"], e.params["max"]
            failures = self.df.filter(f"{e.column} < {lo} OR {e.column} > {hi}").count()
            pass_rate = (self.total_rows - failures) / self.total_rows
            
        elif e.check_type == "row_count_min":
            pass_rate = 1.0 if self.total_rows >= e.params["min"] else 0.0
            failures = 0 if pass_rate == 1.0 else 1
            
        elif e.check_type == "freshness":
            max_val = self.df.agg(max(e.column)).collect()[0][0]
            hours_old = (datetime.now() - max_val).total_seconds() / 3600 if max_val else 9999
            pass_rate = 1.0 if hours_old <= e.params.get("max_hours", 24) else 0.0
            failures = 0 if pass_rate == 1.0 else 1
        else:
            pass_rate = 0.0
            failures = -1
        
        status = "PASS" if pass_rate >= e.params.get("threshold", 0.95) else "FAIL"
        if status == "FAIL" and e.severity == "warning":
            status = "WARN"
        
        result = {
            "expectation": e.name,
            "check_type": e.check_type,
            "column": e.column,
            "severity": e.severity,
            "total_rows": self.total_rows,
            "failures": failures,
            "pass_rate": round(pass_rate, 4),
            "status": status
        }
        self.results.append(result)
        return result
    
    def run_all(self, expectations):
        """Run all expectations and return summary."""
        for e in expectations:
            self.run(e)
        return self.results
    
    def report(self):
        """Print formatted report."""
        passed = sum(1 for r in self.results if r["status"] == "PASS")
        failed = sum(1 for r in self.results if r["status"] == "FAIL")
        warned = sum(1 for r in self.results if r["status"] == "WARN")
        
        print(f"\n{'='*65}")
        print(f"QUALITY REPORT: {self.table_name}")
        print(f"{'='*65}")
        print(f"Total rows: {self.total_rows:,} | Checks: {len(self.results)}")
        print(f"Results: ✅ {passed} PASS | ⚠️ {warned} WARN | ❌ {failed} FAIL")
        print(f"{'='*65}")
        for r in self.results:
            icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}[r["status"]]
            print(f"  {icon} {r['expectation']:<35} {r['pass_rate']:.1%} ({r['failures']} failures)")
        print(f"{'='*65}")

print("✅ Expectation Engine ready")

In [ ]:
# Define and run expectations for orders table
engine = ExpectationEngine("techmart_dw.orders")

expectations = [
    Expectation("order_id_not_null", "not_null", "order_id"),
    Expectation("order_id_unique", "unique", "order_id"),
    Expectation("customer_id_not_null", "not_null", "customer_id"),
    Expectation("amount_positive", "between", "total_amount", {"min": 0, "max": 999999}),
    Expectation("valid_status", "in_set", "status", {"values": ["completed","shipped","processing","pending","cancelled","returned"]}),
    Expectation("valid_payment", "in_set", "payment_method", {"values": ["credit_card","debit_card","paypal","bank_transfer","crypto"]}),
    Expectation("min_row_count", "row_count_min", params={"min": 10000}),
    Expectation("email_completeness", "not_null", "notes", params={"threshold": 0.5}, severity="warning"),
]

engine.run_all(expectations)
engine.report()

In [ ]:
# ============================================
# 🎯 YOUR TURN: Build Expectations for Customers
# ============================================
# Define expectations for techmart_dw.customers:
# 1. customer_id not null
# 2. customer_id unique
# 3. customer_segment in ('Premium','Standard','Basic','New')
# 4. lifetime_value between 0 and 100000
# 5. email completeness > 85% (warning severity)
# Run and print report
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
engine = ExpectationEngine("techmart_dw.customers")
expectations = [
    Expectation("customer_id_not_null", "not_null", "customer_id"),
    Expectation("customer_id_unique", "unique", "customer_id"),
    Expectation("valid_segment", "in_set", "customer_segment", {"values": ["Premium","Standard","Basic","New"]}),
    Expectation("ltv_range", "between", "lifetime_value", {"min": 0, "max": 100000}),
    Expectation("email_completeness", "not_null", "email", params={"threshold": 0.85}, severity="warning"),
]
engine.run_all(expectations)
engine.report()

---
# 📗 Section 3: Quality Gates in Pipelines

In [ ]:
def quality_gate(table_name, expectations, fail_threshold=0.0):
    """Run quality gate — returns True if all critical checks pass.
    
    Args:
        table_name: table to validate
        expectations: list of Expectation objects
        fail_threshold: max allowed FAIL rate (0.0 = zero tolerance)
    """
    engine = ExpectationEngine(table_name)
    engine.run_all(expectations)
    
    critical_failures = [r for r in engine.results if r["status"] == "FAIL" and r["severity"] == "error"]
    fail_rate = len(critical_failures) / len(engine.results) if engine.results else 0
    
    gate_passed = fail_rate <= fail_threshold
    
    icon = "🟢 GATE PASSED" if gate_passed else "🔴 GATE FAILED"
    print(f"\n{icon}: {table_name}")
    print(f"  Critical failures: {len(critical_failures)}/{len(engine.results)}")
    
    if not gate_passed:
        print(f"  Failed checks:")
        for r in critical_failures:
            print(f"    ❌ {r['expectation']}: {r['pass_rate']:.1%}")
    
    return gate_passed

# Run gate on orders
gate_expectations = [
    Expectation("pk_not_null", "not_null", "order_id"),
    Expectation("pk_unique", "unique", "order_id"),
    Expectation("amount_valid", "between", "total_amount", {"min": 0, "max": 999999}),
]

passed = quality_gate("techmart_dw.orders", gate_expectations)
print(f"\nPipeline can proceed: {passed}")

---
# 📗 Section 4: Anomaly Detection

In [ ]:
# Volume anomaly detection
from pyspark.sql.functions import *
import numpy as np

# Daily order volumes
daily_volumes = (spark.table("techmart_dw.orders")
    .groupBy("order_date")
    .count()
    .orderBy("order_date")
    .toPandas()
)

# Calculate statistics
mean_vol = daily_volumes["count"].mean()
std_vol = daily_volumes["count"].std()
threshold_high = mean_vol + 2 * std_vol
threshold_low = mean_vol - 2 * std_vol

# Flag anomalies
daily_volumes["is_anomaly"] = (daily_volumes["count"] > threshold_high) | (daily_volumes["count"] < threshold_low)
anomaly_count = daily_volumes["is_anomaly"].sum()

print(f"Daily Order Volume Analysis:")
print(f"  Mean: {mean_vol:.0f} orders/day")
print(f"  Std: {std_vol:.0f}")
print(f"  Threshold: [{threshold_low:.0f}, {threshold_high:.0f}]")
print(f"  Anomalous days: {anomaly_count} ({anomaly_count/len(daily_volumes)*100:.1f}%)")

---
# 📗 Section 5: Data Contracts

A **data contract** is an agreement between a data producer and consumer:

```yaml
contract:
  name: "orders_silver"
  producer: "ingestion_team"
  consumer: "analytics_team"
  schema:
    - {name: order_id, type: int, nullable: false}
    - {name: total_amount, type: decimal, nullable: false}
    - {name: status, type: string, values: [completed, pending, ...]}
  quality:
    - completeness: 99%
    - uniqueness: 100% on order_id
    - freshness: < 4 hours
  sla:
    - available_by: "06:00 UTC daily"
    - max_latency: "4 hours from source"
```

---
# 🚀 Section 6: Mini Project — Complete Quality Framework

In [ ]:
# Production quality framework: validate multiple tables, store results
import pandas as pd
from datetime import datetime

def run_quality_framework(table_configs):
    """Run quality checks across multiple tables and store results."""
    all_results = []
    
    for table_name, expectations in table_configs.items():
        engine = ExpectationEngine(table_name)
        engine.run_all(expectations)
        
        for r in engine.results:
            r["table"] = table_name
            r["checked_at"] = datetime.now().isoformat()
            all_results.append(r)
    
    # Store results
    results_df = spark.createDataFrame(pd.DataFrame(all_results))
    results_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.quality_framework_results")
    
    # Summary
    pdf = pd.DataFrame(all_results)
    print(f"\n{'='*60}")
    print(f"QUALITY FRAMEWORK SUMMARY")
    print(f"{'='*60}")
    print(f"Tables checked: {pdf['table'].nunique()}")
    print(f"Total checks: {len(pdf)}")
    print(f"Passed: {(pdf['status']=='PASS').sum()}")
    print(f"Warned: {(pdf['status']=='WARN').sum()}")
    print(f"Failed: {(pdf['status']=='FAIL').sum()}")
    print(f"\nPer-table summary:")
    for table in pdf["table"].unique():
        t_results = pdf[pdf["table"] == table]
        passed = (t_results["status"] == "PASS").sum()
        print(f"  {table.split('.')[-1]}: {passed}/{len(t_results)} passed")
    print(f"{'='*60}")

# Define checks for multiple tables
configs = {
    "techmart_dw.orders": [
        Expectation("pk_not_null", "not_null", "order_id"),
        Expectation("pk_unique", "unique", "order_id"),
        Expectation("amount_valid", "between", "total_amount", {"min": 0, "max": 999999}),
        Expectation("valid_status", "in_set", "status", {"values": ["completed","shipped","processing","pending","cancelled","returned"]}),
    ],
    "techmart_dw.customers": [
        Expectation("pk_not_null", "not_null", "customer_id"),
        Expectation("pk_unique", "unique", "customer_id"),
        Expectation("valid_segment", "in_set", "customer_segment", {"values": ["Premium","Standard","Basic","New"]}),
    ],
    "techmart_dw.products": [
        Expectation("pk_not_null", "not_null", "product_id"),
        Expectation("price_positive", "between", "price", {"min": 0, "max": 99999}),
    ],
}

run_quality_framework(configs)

---
# 🏆 Section 7: Interview Questions

## Q1: How do you ensure data quality in production?

1. **Define expectations** per table (completeness, uniqueness, validity)
2. **Quality gates** between pipeline layers (Bronze → Silver → Gold)
3. **Automated monitoring** (track metrics over time, alert on degradation)
4. **Quarantine pattern** (bad records → separate table for review)
5. **Data contracts** (producer guarantees quality to consumer)

## Q2: Six dimensions of data quality?

Completeness, Uniqueness, Validity, Accuracy, Consistency, Timeliness.
Each maps to specific checks you can automate.

## Q3: Fail the pipeline or continue?

**Fail (hard gate):** Critical checks — PK uniqueness, referential integrity.
**Warn (soft gate):** Non-critical — email completeness, optional field nulls.
**Strategy:** Fail on data integrity issues, warn on quality degradation.

## Q4: Data contracts?

Agreement between producer and consumer defining:
- **Schema:** Expected columns and types
- **Quality:** Minimum quality levels (99% completeness)
- **SLA:** Freshness guarantees (available by 6 AM)
- **Enforcement:** Automated validation on every pipeline run

## Q5: Detect quality anomalies?

1. **Volume:** Row count deviates > 2σ from rolling average
2. **Distribution:** Value distribution shifts (KL divergence)
3. **Freshness:** Data older than expected
4. **Null rate:** Sudden spike in NULLs for a column

## Q6: Quality monitoring strategy?

- Store all check results in a Delta table (time-series)
- Dashboard showing pass rates trending over time
- Alert when pass rate drops below threshold
- Weekly quality review meeting with stakeholders

## The Six Dimensions of Data Quality

Data quality is not binary (good/bad). It has six measurable dimensions:

### 1. Completeness
Are all expected values present? No unexpected NULLs?

```sql
-- Completeness check
SELECT
    COUNT(*) AS total_rows,
    COUNT(customer_id) AS non_null_customer_id,
    COUNT(email) AS non_null_email,
    ROUND(COUNT(customer_id) * 100.0 / COUNT(*), 2) AS customer_id_completeness_pct,
    ROUND(COUNT(email) * 100.0 / COUNT(*), 2) AS email_completeness_pct
FROM silver.customers;
```

### 2. Accuracy
Are values correct? Do they match the real world?

```sql
-- Accuracy check: amounts should be positive
SELECT COUNT(*) AS negative_amounts
FROM silver.orders
WHERE total_amount < 0;

-- Accuracy check: dates should be in valid range
SELECT COUNT(*) AS future_orders
FROM silver.orders
WHERE order_date > CURRENT_DATE();
```

### 3. Consistency
Do values agree across systems and tables?

```sql
-- Consistency check: order totals should match sum of line items
SELECT o.order_id,
       o.total_amount AS order_total,
       SUM(oi.line_total) AS calculated_total,
       ABS(o.total_amount - SUM(oi.line_total)) AS discrepancy
FROM silver.orders o
JOIN silver.order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.total_amount
HAVING ABS(o.total_amount - SUM(oi.line_total)) > 0.01;
```

### 4. Timeliness
Is data fresh enough for its intended use?

```sql
-- Timeliness check: data should be updated within 2 hours
SELECT
    MAX(_ingested_at) AS last_ingestion,
    DATEDIFF(HOUR, MAX(_ingested_at), CURRENT_TIMESTAMP()) AS hours_since_update,
    CASE WHEN DATEDIFF(HOUR, MAX(_ingested_at), CURRENT_TIMESTAMP()) > 2
         THEN 'STALE' ELSE 'FRESH' END AS freshness_status
FROM silver.orders;
```

### 5. Validity
Do values conform to defined rules and formats?

```sql
-- Validity check: email format
SELECT COUNT(*) AS invalid_emails
FROM silver.customers
WHERE email NOT RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$';

-- Validity check: status values
SELECT status, COUNT(*) AS count
FROM silver.orders
WHERE status NOT IN ('pending', 'completed', 'shipped', 'cancelled')
GROUP BY status;
```

### 6. Uniqueness
Are there unexpected duplicates?

```sql
-- Uniqueness check: order_id should be unique
SELECT order_id, COUNT(*) AS occurrences
FROM silver.orders
GROUP BY order_id
HAVING COUNT(*) > 1;
```

## Quality Check Implementation Patterns

### Pattern 1: SQL-Based Checks (Simplest)

```sql
-- Run as part of your pipeline, fail if any check fails
CREATE OR REPLACE TEMP VIEW quality_results AS
SELECT 'null_order_id' AS check_name,
       COUNT(*) AS failures,
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM silver.orders WHERE order_id IS NULL
UNION ALL
SELECT 'negative_amount',
       COUNT(*),
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM silver.orders WHERE total_amount < 0
UNION ALL
SELECT 'duplicate_orders',
       COUNT(*) - COUNT(DISTINCT order_id),
       CASE WHEN COUNT(*) = COUNT(DISTINCT order_id) THEN 'PASS' ELSE 'FAIL' END
FROM silver.orders;

-- Fail pipeline if any check fails
SELECT * FROM quality_results WHERE status = 'FAIL';
```

### Pattern 2: Python Class (Reusable)

```python
class DataQualityChecker:
    def __init__(self, df, table_name):
        self.df = df
        self.table_name = table_name
        self.results = []
    
    def not_null(self, column, threshold_pct=0):
        null_count = self.df.filter(col(column).isNull()).count()
        total = self.df.count()
        null_pct = null_count / total * 100
        passed = null_pct <= threshold_pct
        self.results.append({
            "check": f"not_null({column})",
            "passed": passed,
            "details": f"{null_pct:.2f}% nulls (threshold: {threshold_pct}%)"
        })
        return self
    
    def unique(self, column):
        total = self.df.count()
        distinct = self.df.select(column).distinct().count()
        passed = total == distinct
        self.results.append({
            "check": f"unique({column})",
            "passed": passed,
            "details": f"{total - distinct} duplicates found"
        })
        return self
    
    def in_range(self, column, min_val, max_val):
        violations = self.df.filter(
            (col(column) < min_val) | (col(column) > max_val)
        ).count()
        passed = violations == 0
        self.results.append({
            "check": f"in_range({column}, {min_val}, {max_val})",
            "passed": passed,
            "details": f"{violations} values out of range"
        })
        return self
    
    def assert_all_pass(self):
        failed = [r for r in self.results if not r["passed"]]
        if failed:
            raise ValueError(f"Quality checks failed: {failed}")
        return True
```

### Pattern 3: DLT Expectations (Declarative)

```python
# In a DLT pipeline -- quality is built into the pipeline definition
@dlt.table
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "total_amount > 0")
@dlt.expect("valid_status", "status IN ('pending','completed','shipped','cancelled')")
def silver_orders():
    return dlt.read_stream("bronze_orders")
```

In [ ]:
# Comprehensive data quality framework
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, when, current_timestamp, lit

class DataQualityFramework:
    """Production-grade data quality framework."""
    
    def __init__(self, spark_session, table_name):
        self.spark = spark_session
        self.table_name = table_name
        self.df = spark_session.table(table_name)
        self.checks = []
        self.results = []
    
    def not_null(self, column, threshold_pct=0, severity="error"):
        """Check that a column has fewer NULLs than threshold."""
        self.checks.append({
            "type": "not_null", "column": column,
            "threshold_pct": threshold_pct, "severity": severity
        })
        return self
    
    def unique(self, column, severity="error"):
        """Check that a column has no duplicates."""
        self.checks.append({"type": "unique", "column": column, "severity": severity})
        return self
    
    def in_set(self, column, allowed_values, severity="error"):
        """Check that all values are in the allowed set."""
        self.checks.append({
            "type": "in_set", "column": column,
            "allowed_values": allowed_values, "severity": severity
        })
        return self
    
    def positive(self, column, severity="error"):
        """Check that all values are positive."""
        self.checks.append({"type": "positive", "column": column, "severity": severity})
        return self
    
    def row_count_between(self, min_rows, max_rows, severity="error"):
        """Check that row count is within expected range."""
        self.checks.append({
            "type": "row_count", "min": min_rows, "max": max_rows, "severity": severity
        })
        return self
    
    def run(self):
        """Execute all checks and return results."""
        total = self.df.count()
        self.results = []
        
        for check in self.checks:
            result = {"check_type": check["type"], "severity": check["severity"]}
            
            if check["type"] == "not_null":
                null_count = self.df.filter(col(check["column"]).isNull()).count()
                null_pct = null_count / total * 100 if total > 0 else 0
                passed = null_pct <= check["threshold_pct"]
                result.update({
                    "column": check["column"],
                    "passed": passed,
                    "metric": f"{null_pct:.2f}% nulls",
                    "threshold": f"<= {check['threshold_pct']}%"
                })
            
            elif check["type"] == "unique":
                distinct = self.df.select(check["column"]).distinct().count()
                duplicates = total - distinct
                passed = duplicates == 0
                result.update({
                    "column": check["column"],
                    "passed": passed,
                    "metric": f"{duplicates} duplicates",
                    "threshold": "0 duplicates"
                })
            
            elif check["type"] == "in_set":
                violations = self.df.filter(
                    ~col(check["column"]).isin(check["allowed_values"])
                ).count()
                passed = violations == 0
                result.update({
                    "column": check["column"],
                    "passed": passed,
                    "metric": f"{violations} invalid values",
                    "threshold": f"in {check['allowed_values']}"
                })
            
            elif check["type"] == "positive":
                violations = self.df.filter(col(check["column"]) <= 0).count()
                passed = violations == 0
                result.update({
                    "column": check["column"],
                    "passed": passed,
                    "metric": f"{violations} non-positive values",
                    "threshold": "> 0"
                })
            
            elif check["type"] == "row_count":
                passed = check["min"] <= total <= check["max"]
                result.update({
                    "passed": passed,
                    "metric": f"{total:,} rows",
                    "threshold": f"{check['min']:,} - {check['max']:,}"
                })
            
            self.results.append(result)
        
        return self
    
    def report(self):
        """Print quality report."""
        errors = [r for r in self.results if not r["passed"] and r["severity"] == "error"]
        warnings = [r for r in self.results if not r["passed"] and r["severity"] == "warning"]
        passed = [r for r in self.results if r["passed"]]
        
        print(f"\n📊 DATA QUALITY REPORT: {self.table_name}")
        print("=" * 60)
        print(f"   Total checks: {len(self.results)}")
        print(f"   Passed: {len(passed)}")
        print(f"   Errors: {len(errors)}")
        print(f"   Warnings: {len(warnings)}")
        
        for r in self.results:
            status = "✅" if r["passed"] else ("❌" if r["severity"] == "error" else "⚠️")
            col_info = f" [{r.get('column', 'table')}]"
            print(f"   {status} {r['check_type']}{col_info}: {r.get('metric', '')} (threshold: {r.get('threshold', '')})")
        
        if errors:
            raise ValueError(f"Quality check FAILED: {len(errors)} error(s) found")
        return self


# Run quality checks on orders table
print("Running quality checks on techmart_dw.orders...")
try:
    qc = (DataQualityFramework(spark, "techmart_dw.orders")
        .not_null("order_id", threshold_pct=0, severity="error")
        .not_null("customer_id", threshold_pct=0, severity="error")
        .not_null("total_amount", threshold_pct=0, severity="error")
        .unique("order_id", severity="error")
        .positive("total_amount", severity="error")
        .in_set("status", ["pending", "completed", "shipped", "cancelled"], severity="warning")
        .row_count_between(min_rows=1000, max_rows=100000, severity="error")
        .run()
        .report())
    print("\n✅ All quality checks passed!")
except ValueError as e:
    print(f"\n❌ Quality check failed: {e}")


## Quality Monitoring in Production

### Quality Metrics to Track Over Time

Track quality metrics as time series to detect TRENDS:

```
Date        | null_rate | duplicate_rate | freshness_hours | row_count
2024-03-10  | 0.1%      | 0.0%           | 1.2             | 18,432
2024-03-11  | 0.1%      | 0.0%           | 1.1             | 19,105
2024-03-12  | 0.1%      | 0.0%           | 1.3             | 18,876
2024-03-13  | 2.5%      | 0.0%           | 1.2             | 18,543  ← SPIKE!
2024-03-14  | 0.1%      | 0.0%           | 1.1             | 18,901
```

A sudden spike in null_rate on 2024-03-13 indicates a source system issue.

### Alerting Strategy

| Metric | Warning Threshold | Critical Threshold | Action |
|--------|------------------|-------------------|--------|
| NULL rate | > 1% | > 5% | Investigate source |
| Duplicate rate | > 0.1% | > 1% | Check ingestion |
| Freshness | > 2 hours | > 4 hours | Check pipeline |
| Row count change | > 20% | > 50% | Investigate source |
| Failed checks | > 0 | > 5 | Page on-call |

### Quality Score

Aggregate all checks into a single quality score:

```python
def calculate_quality_score(check_results):
    weights = {"completeness": 0.3, "uniqueness": 0.2, "validity": 0.2,
               "consistency": 0.15, "timeliness": 0.15}
    
    scores = {}
    for dimension, checks in check_results.items():
        passed = sum(1 for c in checks if c["passed"])
        scores[dimension] = passed / len(checks) * 100
    
    weighted_score = sum(scores.get(dim, 100) * weight
                        for dim, weight in weights.items())
    return weighted_score
```

In [ ]:
# Quality monitoring over time
from datetime import datetime, timedelta
import random

class QualityMonitor:
    """Tracks quality metrics over time and detects anomalies."""
    
    def __init__(self, table_name, alert_thresholds):
        self.table_name = table_name
        self.thresholds = alert_thresholds
        self.history = []
    
    def record_metrics(self, date, metrics):
        """Record quality metrics for a date."""
        self.history.append({"date": date, **metrics})
    
    def detect_anomalies(self):
        """Detect metrics that exceed thresholds."""
        alerts = []
        for record in self.history:
            for metric, value in record.items():
                if metric == "date":
                    continue
                if metric in self.thresholds:
                    warn_threshold = self.thresholds[metric].get("warning")
                    crit_threshold = self.thresholds[metric].get("critical")
                    if crit_threshold and value > crit_threshold:
                        alerts.append({
                            "date": record["date"],
                            "metric": metric,
                            "value": value,
                            "severity": "CRITICAL",
                            "threshold": crit_threshold
                        })
                    elif warn_threshold and value > warn_threshold:
                        alerts.append({
                            "date": record["date"],
                            "metric": metric,
                            "value": value,
                            "severity": "WARNING",
                            "threshold": warn_threshold
                        })
        return alerts
    
    def show_trend(self):
        """Show quality metrics trend."""
        print(f"\n📈 QUALITY TREND: {self.table_name}")
        print(f"{'Date':<12} {'Null%':<8} {'Dup%':<8} {'Fresh(h)':<10} {'Rows':<10} {'Status'}")
        print("-" * 60)
        for record in self.history:
            null_pct = record.get("null_rate_pct", 0)
            dup_pct = record.get("duplicate_rate_pct", 0)
            fresh = record.get("freshness_hours", 0)
            rows = record.get("row_count", 0)
            
            # Determine status
            if null_pct > 5 or dup_pct > 1 or fresh > 4:
                status = "❌ CRITICAL"
            elif null_pct > 1 or dup_pct > 0.1 or fresh > 2:
                status = "⚠️ WARNING"
            else:
                status = "✅ OK"
            
            print(f"{record['date']:<12} {null_pct:<8.2f} {dup_pct:<8.3f} {fresh:<10.1f} {rows:<10,} {status}")


# Simulate 7 days of quality metrics
random.seed(42)
monitor = QualityMonitor(
    "silver.orders",
    alert_thresholds={
        "null_rate_pct": {"warning": 1.0, "critical": 5.0},
        "duplicate_rate_pct": {"warning": 0.1, "critical": 1.0},
        "freshness_hours": {"warning": 2.0, "critical": 4.0},
    }
)

base_date = datetime(2024, 3, 10)
for i in range(7):
    date = (base_date + timedelta(days=i)).strftime("%Y-%m-%d")
    # Inject an anomaly on day 3
    null_rate = 2.5 if i == 3 else round(random.uniform(0.05, 0.15), 2)
    monitor.record_metrics(date, {
        "null_rate_pct": null_rate,
        "duplicate_rate_pct": round(random.uniform(0.0, 0.05), 3),
        "freshness_hours": round(random.uniform(0.8, 1.5), 1),
        "row_count": random.randint(18000, 20000),
    })

monitor.show_trend()

alerts = monitor.detect_anomalies()
if alerts:
    print(f"\n🚨 ALERTS DETECTED:")
    for alert in alerts:
        print(f"   {alert['severity']}: {alert['date']} -- {alert['metric']} = {alert['value']} (threshold: {alert['threshold']})")


In [ ]:
# ============================================
# 🎯 YOUR TURN: Build a Quality Framework
# ============================================
# Using the DataQualityFramework class above, run quality checks on:
# 1. techmart_dw.customers
#    - customer_id: not null, unique
#    - email: not null (allow 5% nulls -- some customers don't provide email)
#    - customer_segment: in set ('bronze', 'silver', 'gold', 'platinum')
#    - lifetime_value: positive
#    - Row count: between 4000 and 6000
#
# 2. techmart_dw.order_items
#    - order_item_id: not null, unique
#    - quantity: positive
#    - unit_price: positive
#    - line_total: positive
#
# Write your quality checks below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
print("QUALITY CHECKS: techmart_dw.customers")
print("=" * 60)
try:
    qc_customers = (DataQualityFramework(spark, "techmart_dw.customers")
        .not_null("customer_id", threshold_pct=0, severity="error")
        .unique("customer_id", severity="error")
        .not_null("email", threshold_pct=5.0, severity="warning")
        .in_set("customer_segment", ["bronze", "silver", "gold", "platinum"], severity="warning")
        .positive("lifetime_value", severity="error")
        .row_count_between(min_rows=4000, max_rows=6000, severity="error")
        .run()
        .report())
    print("\n✅ Customers quality checks passed!")
except ValueError as e:
    print(f"\n❌ {e}")

print("\n\nQUALITY CHECKS: techmart_dw.order_items")
print("=" * 60)
try:
    qc_items = (DataQualityFramework(spark, "techmart_dw.order_items")
        .not_null("order_item_id", threshold_pct=0, severity="error")
        .unique("order_item_id", severity="error")
        .positive("quantity", severity="error")
        .positive("unit_price", severity="error")
        .positive("line_total", severity="error")
        .run()
        .report())
    print("\n✅ Order items quality checks passed!")
except ValueError as e:
    print(f"\n❌ {e}")


---
# ✅ Notebook Complete!

**What was covered:**
- Six dimensions of data quality with examples
- Complete ExpectationEngine (not_null, unique, in_set, between, row_count, freshness)
- Quality gates with pass/fail logic
- Volume anomaly detection (statistical thresholds)
- Data contracts (schema + quality + SLA)
- Mini project: multi-table quality framework with Delta storage
- 6 interview questions

**Next:** Notebook 28 — CI/CD & Version Control

---